# Продовження навчання з чекпоінта

Завантажує модель і `VecNormalize` з чекпоінта і доводить навчання до `TOTAL_STEPS`.
`config.py` і `hexapod_env.py` не мають змінюватися з моменту старту цього запуску, інакше друга частина піде на іншій нагороді.

In [7]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

from hexapod_env import HexapodEnv

In [8]:
N_ENVS = 8
TOTAL_STEPS = 5_000_000
CKPT = 3_000_000
RUN_NAME = "stage3-stroke-air"
CKPT_DIR = f"models/{RUN_NAME}/checkpoints"

In [9]:
def make_env():
    return Monitor(HexapodEnv())

vec_env = SubprocVecEnv([make_env for _ in range(N_ENVS)])
vec_env = VecNormalize.load(f"{CKPT_DIR}/rl_model_vecnormalize_{CKPT}_steps.pkl", vec_env)
vec_env.training = True       # load за замовчуванням вимикає оновлення статистики
vec_env.norm_reward = True

In [10]:
INFO_KEYS = (
    "forward_vel",
    "side_vel",
    "reward",
    "r_vel",
    "r_gait",
    "r_yaw",
    "r_vy",
    "r_air",
    "pen_roll",
    "pen_pitch",
    "pen_stroke",
    "pen_rate",
)

class InfoCallback(BaseCallback):
    def _on_step(self):
        for info in self.locals["infos"]:
            for k in INFO_KEYS:
                self.logger.record_mean("env/" + k, info[k])
        return True

In [11]:
lr = lambda progress: 5e-5 * progress / 0.4

model = PPO.load(
    f"{CKPT_DIR}/rl_model_{CKPT}_steps",
    env=vec_env,
    device="cpu",
    tensorboard_log="logs/",
    custom_objects={"learning_rate": lr, "lr_schedule": lr},
)
print("кроків у чекпоінті:", model.num_timesteps)

checkpoint = CheckpointCallback(
    save_freq=100_000 // N_ENVS,
    save_path=CKPT_DIR,
    save_vecnormalize=True,
)

кроків у чекпоінті: 3000000


In [12]:
model.learn(
    total_timesteps=TOTAL_STEPS - CKPT,
    callback=[checkpoint, InfoCallback()],
    tb_log_name=RUN_NAME,
    reset_num_timesteps=False,
)

model.save(f"models/{RUN_NAME}/model")
vec_env.save(f"models/{RUN_NAME}/vecnormalize.pkl")
vec_env.close()

Logging to logs/stage3-stroke-air_1
height=0.1146, gravity_z=-1.0000, terminated=False, truncated=True
height=0.1189, gravity_z=-0.9999, terminated=False, truncated=True
height=0.1152, gravity_z=-0.9996, terminated=False, truncated=True
height=0.1135, gravity_z=-0.9996, terminated=False, truncated=True
height=0.1132, gravity_z=-0.9999, terminated=False, truncated=True
height=0.1171, gravity_z=-0.9998, terminated=False, truncated=True
height=0.1190, gravity_z=-0.9998, terminated=False, truncated=True
height=0.1126, gravity_z=-0.9997, terminated=False, truncated=True
----------------------------------
| env/               |           |
|    forward_vel     | 0.17      |
|    pen_pitch       | 0.00016   |
|    pen_rate        | 3.41      |
|    pen_roll        | 0.000134  |
|    pen_stroke      | 0.000507  |
|    r_air           | 0.0194    |
|    r_gait          | 0.904     |
|    r_vel           | 0.969     |
|    r_vy            | 0.971     |
|    r_yaw           | 0.951     |
|    rew